# PyTorch-Optimizer-Visualisierung (Plotly)

Animierter Vergleich der Optimierungspfade von **SGD, Adam, AdamW und RMSprop** auf der Rosenbrock-Funktion.

Diese Variante nutzt **Plotly** statt ipympl/`%matplotlib widget`: Die Animation laeuft vollstaendig im Browser-JavaScript und rendert daher auch in **VS Code** zuverlaessig — ohne Kernel-Callbacks, ohne ipympl.

Die vier Optimizer starten gemeinsam am selben Punkt und laufen gleichzeitig zum Minimum (1, 1). Links die Pfade im Parameterraum, rechts der mitlaufende Loss.


In [ ]:
%pip install plotly torch numpy -q

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

START = (-1.5, 1.5)
MAX_STEPS = 200
STRIDE = 2                      # nur jeden 2. Schritt als Frame -> kleinere Datei

# Loss-Gelaende (Hintergrund)
x_lin = np.linspace(-2.0, 2.0, 120)
y_lin = np.linspace(-1.0, 3.0, 120)
Xg, Yg = np.meshgrid(x_lin, y_lin)
Zg = rosenbrock(Xg, Yg)

# Pro Optimizer eine brauchbare Lernrate (Rosenbrock ist heikel - SGD vertraegt nur winzige LR)
COLORS = {'SGD': '#1f77b4', 'Adam': '#d62728', 'AdamW': '#2ca02c', 'RMSprop': '#9467bd'}
CONFIG = {
    'SGD':     (2e-3, dict(momentum=0.9)),
    'Adam':    (2e-1, dict()),
    'AdamW':   (2e-1, dict(weight_decay=0.01)),
    'RMSprop': (2e-2, dict(alpha=0.99)),
}

def trajectory(name, lr, kwargs):
    """Berechnet den Pfad eines Optimizers; bricht bei Divergenz sauber ab."""
    torch.manual_seed(42)
    p = torch.tensor([START[0], START[1]], requires_grad=True)
    opt = getattr(torch.optim, name)([p], lr=lr, **kwargs)
    start_loss = float(rosenbrock(torch.tensor(START[0]), torch.tensor(START[1])))
    tx, ty, lh = [START[0]], [START[1]], [start_loss]
    for _ in range(MAX_STEPS):
        opt.zero_grad()
        loss = rosenbrock(p[0], p[1])
        if not torch.isfinite(loss):           # divergiert -> abbrechen
            break
        loss.backward()
        opt.step()
        if not torch.isfinite(p).all():        # divergiert -> abbrechen
            break
        tx.append(p[0].item())
        ty.append(p[1].item())
        lh.append(loss.item())
    return tx, ty, lh

traj = {n: trajectory(n, lr, kw) for n, (lr, kw) in CONFIG.items()}
names = list(CONFIG)

for n in names:
    tx, ty, lh = traj[n]
    print(f"{n:8s}: {len(tx):3d} Schritte | End-Loss {lh[-1]:.4f} bei ({tx[-1]:.2f}, {ty[-1]:.2f})")


In [ ]:
fig = make_subplots(
    rows=1, cols=2, column_widths=[0.58, 0.42],
    subplot_titles=('Optimierungspfade auf Rosenbrock', 'Loss-Verlauf (log)'),
    horizontal_spacing=0.09,
)

# --- statischer Hintergrund: Loss-Gelaende (log-skaliert, damit das Tal sichtbar ist) ---
fig.add_trace(go.Contour(
    x=x_lin, y=y_lin, z=np.log10(Zg + 1), colorscale='Viridis', showscale=False,
    ncontours=22, contours=dict(coloring='fill'),
    line=dict(width=0.4, color='rgba(0,0,0,0.25)'), hoverinfo='skip'), 1, 1)
fig.add_trace(go.Scatter(x=[1], y=[1], mode='markers', name='Minimum (1,1)',
    marker=dict(symbol='star', size=16, color='white', line=dict(color='black', width=1))), 1, 1)
fig.add_trace(go.Scatter(x=[START[0]], y=[START[1]], mode='markers', name='Start',
    marker=dict(size=11, color='lime', line=dict(color='black', width=1))), 1, 1)

PATH0 = len(fig.data)            # Index der ersten Pfad-Spur
for n in names:
    fig.add_trace(go.Scatter(x=[traj[n][0][0]], y=[traj[n][1][0]], mode='lines+markers',
        name=n, line=dict(color=COLORS[n], width=2.5), marker=dict(size=4, color=COLORS[n])), 1, 1)
for n in names:
    fig.add_trace(go.Scatter(x=[0], y=[traj[n][2][0]], mode='lines',
        name=n, line=dict(color=COLORS[n], width=2), showlegend=False), 1, 2)

# --- Animations-Frames: bei Schritt s alle Pfade bis s zeigen ---
maxlen = max(len(traj[n][0]) for n in names)
steps = list(range(0, maxlen, STRIDE)) + [maxlen - 1]
frames = []
for s in steps:
    data = []
    for n in names:                          # Pfade
        tx, ty, _ = traj[n]; k = min(s, len(tx) - 1)
        data.append(go.Scatter(x=tx[:k+1], y=ty[:k+1]))
    for n in names:                          # Loss-Kurven
        _, _, lh = traj[n]; k = min(s, len(lh) - 1)
        data.append(go.Scatter(x=list(range(k+1)), y=lh[:k+1]))
    frames.append(go.Frame(data=data, traces=list(range(PATH0, PATH0 + 8)), name=str(s)))
fig.frames = frames

fig.update_layout(
    height=560, margin=dict(t=70), legend=dict(x=0.42, y=0.98),
    updatemenus=[dict(type='buttons', showactive=False, x=0.0, y=1.15, xanchor='left', buttons=[
        dict(label='\u25b6 Play', method='animate',
             args=[None, dict(frame=dict(duration=60, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label='\u23f8 Pause', method='animate',
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])])],
    sliders=[dict(active=0, x=0.0, len=0.55, currentvalue=dict(prefix='Schritt: '),
        steps=[dict(method='animate', label=str(s),
            args=[[str(s)], dict(mode='immediate', frame=dict(duration=0, redraw=True))]) for s in steps])],
)
fig.update_xaxes(range=[-2, 2], title='x', row=1, col=1)
fig.update_yaxes(range=[-1, 3], title='y', row=1, col=1)
fig.update_xaxes(title='Iteration', row=1, col=2)
fig.update_yaxes(type='log', title='Loss', row=1, col=2)

fig.show()


## Hinweise

- **Play / Pause / Schritt-Slider** sind Plotly-nativ und laufen im Browser — keine Kernel-Kommunikation noetig, daher VS-Code-tauglich.
- Die vier Optimizer nutzen **unterschiedliche Lernraten** (siehe `CONFIG`), weil Rosenbrock sehr unterschiedlich gut zu jedem passt. SGD divergiert bei zu grosser LR sofort; die `trajectory`-Funktion faengt das ueber `torch.isfinite` ab.
- Beobachtung fuer den Unterricht: Mit gut gewaehlter kleiner LR konvergiert **SGD+Momentum** auf dieser glatten Landschaft deutlich schneller als Adam/RMSprop — ein schoenes Gegenbeispiel zur Faustregel \"Adam ist immer am besten\".
- Lernraten/Parameter aendern: Werte in `CONFIG` anpassen und beide Code-Zellen erneut ausfuehren.
